Notebook purpose:
**Objective**
This notebook standardizes and validates consumption and product delivery data for downstream analysis and database ingestion. It addresses data quality issues, ensures consistency across datasets, and exports clean, production-ready CSV files.

**Key Tasks**
1. Data Quality Validation: Sanity checks on structure, shape, and data types
2. Product Name Standardization: Fix misspellings and inconsistent naming conventions
3. Data Reconciliation: Resolve missing product attributes via cross-referencing between datasets
4. Enrichment: Add product categories and unique identifiers (load_id, record_id)
5. Normalized data: normalized to `YYYY-MM-DD` format for database consistency
6. Export: Serialize to CSV with ISO 8601 date formatting for SQL compatibility

Notebook purpose:

In [2]:
import pandas as pd
import yaml

### Reading datasets.

In [ ]:
with open("../config/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

In [4]:
# Reading consumption dataset.
consumption_merged_ready = pd.read_excel(config["consumption_dataset_ready_path"], engine="openpyxl")

In [5]:
# Reading products dataset.
products_years = pd.read_csv(config["products_years_path"], encoding="utf-8")

---
---

### Sanity check.

In [6]:
consumption_merged_ready.info()
consumption_merged_ready.shape

<class 'pandas.DataFrame'>
RangeIndex: 19843 entries, 0 to 19842
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   date            19843 non-null  datetime64[us]
 1   plant           19843 non-null  str           
 2   product         19843 non-null  str           
 3   note            4258 non-null   str           
 4   volume          19843 non-null  float64       
 5   cement_42_5r    19843 non-null  float64       
 6   cement_3a_42    19843 non-null  float64       
 7   cement_4av      19843 non-null  float64       
 8   arid_04         19843 non-null  float64       
 9   arid_8_14       19843 non-null  float64       
 10  arid_12_20      19843 non-null  float64       
 11  water           19843 non-null  float64       
 12  ad_complast_mr  19843 non-null  float64       
 13  ad_dynamon_sx   19843 non-null  float64       
 14  ad_aut_dynamon  19843 non-null  float64       
 15  ad_auramix   

(19843, 17)

In [7]:
products_years.info()
products_years.shape

<class 'pandas.DataFrame'>
RangeIndex: 9012 entries, 0 to 9011
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer          9012 non-null   str    
 1   worksite          9012 non-null   str    
 2   plant             9012 non-null   str    
 3   product           9012 non-null   str    
 4   product_category  9012 non-null   str    
 5   volume            9012 non-null   float64
 6   billing_company   9012 non-null   str    
 7   date              9012 non-null   str    
dtypes: float64(1), str(7)
memory usage: 563.4 KB


(9012, 8)

date is str in products_years, but datetime in consumption_merged_ready. We need to convert date in products_years to datetime.
However, I decided to finally convert it just for the exportation for the following reasons:
1. In some ahead steps, the dataframe became again to the original format.
2. There is no need to convert the format now.
*Note in "learn_by_issues" file about how pandas reads date data types and how to convert it.*

---

##### 1. Verify products names in both dataframes.

In [8]:
products_in_prod = set(products_years["product"].dropna().unique())
products_in_cons = set(consumption_merged_ready["product"].dropna().unique())
all_products = sorted(products_in_prod | products_in_cons)

product_comparison = pd.DataFrame({
    "product": all_products,
    "in_products_years": ["✔" if p in products_in_prod else "" for p in all_products],
    "in_consumption": ["✔" if p in products_in_cons else "" for p in all_products],
    "match": ["✔" if (p in products_in_prod and p in products_in_cons) else "✘" for p in all_products],
})

product_comparison

,product,in_products_years,in_consumption,match
0,C40-50 XC4,✔,,✘
1,C45-55 XC4,✔,,✘
2,CT-C12-F3,✔,✔,✔
3,CT-C16-F3,✔,✔,✔
4,D-100-B-15,,✔,✘
...,...,...,...,...
133,arid-04,✔,✔,✔
134,arid-15,,✔,✘
135,arid-20,✔,✔,✔
136,arid-25,✔,,✘


In [9]:
# Once I had a look, we don't need to export this df again.
# product_comparison.to_excel("C:\\Users\\User\\Desktop\\Data Projects\\Workspaces\\consumption_and_product_workspace\\consumption_and_products_project\\data\\processed\\product_comparison.xlsx", index=False)

---

#### 2. Fixing the misspelling product name in datasets.

*To correct*

Consumption dataset:
H-100/B/20: H-100-B-20

HA-25-F-15-IIA: HA-25-F-15

HF-35-B-20: HF-3.5-B-20

HF-35-B-20-XC2: HF-3.5-B-20-XC2

Products dataset:
C40-50 XC4: C40-50-XC4

C45-55 XC4: C45-55-XC4

H-175-B-25: H-175-B-20

HA-25-F-15-IIA: HA-25-F-15. 
Missing decimal point in concrete resistance codes (`HF` family) caused a tenfold overstatement of theoretical resistance.

In [10]:
# Product names corrections in consumption_merged_ready dataset.
consumption_name_map = {
"H-100/B/20 ": "H-100-B-20",
"HA-25-F-15-IIA": "HA-25-F-15",
"HF-35-B-20": "HF-3.5-B-20",
"HF-35-B-20-XC2": "HF-3.5-B-20-XC2",
}

consumption_merged_ready["product"] = consumption_merged_ready["product"].replace(consumption_name_map)

In [11]:
# Verify they are all correct
for p in sorted(consumption_merged_ready["product"].unique()):
    print(p)

CT-C12-F3
CT-C16-F3
D-100-B-15
D-100-B-20
D-200-P-15
D-200-S-15
D-200-S-20
D-250-B-20
D-250-F-15
D-250-S-15
D-250-S-20
D-350-L-4
D-350-S-20
D-400-B-4
D-400-P-4
D-450-P-4
H-100-B-20
H-15-B-20
H-150-B-15
H-150-B-20
H-175-B-15
H-175-B-20
H-200-B-20
H-250-B-15
H-250-B-20
HA-25-B-15
HA-25-B-15-X0
HA-25-B-15-XC1
HA-25-B-15-XC2
HA-25-B-15-XC3
HA-25-B-20
HA-25-B-20-X0
HA-25-B-20-XC1
HA-25-B-20-XC2
HA-25-B-20-XC3
HA-25-B-20-XC4
HA-25-F-15
HA-25-F-15-X0
HA-25-F-15-XC1
HA-25-F-15-XC2
HA-25-F-15-XC4
HA-25-F-20
HA-25-F-20-X0
HA-25-F-20-XC1
HA-25-F-20-XC2
HA-25-L-15
HA-30-B-15
HA-30-B-15-XA1
HA-30-B-15-XC1
HA-30-B-15-XC2
HA-30-B-20
HA-30-B-20-IV
HA-30-B-20-IV-QB
HA-30-B-20-QB
HA-30-B-20-XA1
HA-30-B-20-XC1
HA-30-B-20-XC2
HA-30-B-20-XC3
HA-30-B-20-XC4
HA-30-B-20-XD2
HA-30-B-20-XS1
HA-30-F-15
HA-30-F-15-IV-QB
HA-30-F-15-X0
HA-30-F-15-XC1
HA-30-F-15-XC2
HA-30-F-15-XC3
HA-30-F-15-XC4
HA-30-F-15-XC4-XF1
HA-30-F-20
HA-30-F-20-X0
HA-30-F-20-XC1
HA-30-F-20-XC2
HA-30-F-20-XC3
HA-30-F-20-XC4-XF1
HA-30-F-20-XD3

In [12]:
# Product names corrections in product_years dataset.
product_name_map = {
"C40-50 XC4": "C40-50-XC4",
"C45-55 XC4": "C45-55-XC4",
"H-100/B/20": "H-100-B-20",
"HA-25-F-15-IIA": "HA-25-F-15",
"HF-35-B-20": "HF-3.5-B-20",
"HF-35-B-20-XC2": "HF-3.5-B-20-XC2",
}

products_years["product"] = products_years["product"].replace(product_name_map)

In [13]:
# Verify they are all correct
for p in sorted(products_years["product"].unique()):
    print(p)

C40-50-XC4
C45-55-XC4
CT-C12-F3
CT-C16-F3
D-200
D-200-P-15
D-200-S-15
D-200-S-20
D-250-B-15
D-250-F-15
D-250-S-15
D-250-S-20
D-350
D-350-S-20
D-400-B-14
D-450-B-14
H-100-B-15
H-100-B-20
H-100-B-25
H-150-B-15
H-150-B-20
H-175-B-15
H-175-B-20
H-175-B-25
H-250-B-15
HA-25-B-15
HA-25-B-15-X0
HA-25-B-15-XC1
HA-25-B-15-XC2
HA-25-B-20
HA-25-B-20-X0
HA-25-B-20-XA1
HA-25-B-20-XC1
HA-25-B-20-XC2
HA-25-B-20-XC3
HA-25-B-20-XC4
HA-25-F-15
HA-25-F-15-X0
HA-25-F-15-XC1
HA-25-F-15-XC2
HA-25-F-15-XC4
HA-25-F-20
HA-25-F-20-X0
HA-25-F-20-XC1
HA-25-F-20-XC2
HA-25-F-20-XC3
HA-25-L-15
HA-30-B-15
HA-30-B-15-XC1
HA-30-B-15-XC2
HA-30-B-20
HA-30-B-20-IV-QB
HA-30-B-20-QB
HA-30-B-20-XA1
HA-30-B-20-XC1
HA-30-B-20-XC2
HA-30-B-20-XC3
HA-30-B-20-XC4
HA-30-B-20-XD2
HA-30-B-20-XS1
HA-30-B-IV-QB
HA-30-F-15
HA-30-F-15-QB
HA-30-F-15-X0
HA-30-F-15-XC1
HA-30-F-15-XC2
HA-30-F-15-XC3
HA-30-F-15-XC4
HA-30-F-15-XC4-XA1
HA-30-F-15-XC4-XF1
HA-30-F-20
HA-30-F-20-X0
HA-30-F-20-XC1
HA-30-F-20-XC2
HA-30-F-20-XC3
HA-30-F-20-XC4
HA-30-F

---

#### 3. Exploring and solving product attributes/dimensions.

The consumption dataset has no anomalies in this dimension.

The products dataset contains four entries requiring investigation:

- **`no_registered`** — appears only once (1 row out of 9,012). Too ambiguous to resolve via cross-reference; treated as a valid product count.
- **`HA-30-B-IV-QB`** — data entry error: the arid dimension was missing. Cross-referencing with the consumption dataset identified the correct value; corrected to `HA-30-B-20-IV-QB`.
- **`HM-20-15`** — data entry error: the consistency dimension was missing. Same resolution approach as above.
- **`MD`** — treated as a distinct product category rather than imputing from other MD variants, to avoid distorting the distribution.

In [14]:
# Checking for "no_registered" products in products_years dataset.
count = (products_years["product"] == "no_registered").sum()
total = len(products_years)
print(f"no_registered: {count} rows out of {total} ({count / total * 100:.2f}%)")
products_years[products_years["product"] == "no_registered"]

no_registered: 1 rows out of 9012 (0.01%)


,customer,worksite,plant,product,product_category,volume,billing_company,date
2322,NORCIA_XENOFONTE,PRAENESTE_URIA_URIA_CLUSIUM_ZADAR,P_A,no_registered,uncategorized,7.0,H,2024-06-28


Even though not all loads are registered in the consumption dataset, if this load is registered, we can obtain the missing product information by checking its consumption. As common fields, we have the date, volume, and plant (just the zone).

In [15]:
# Display all rows in consumption_merged_ready matching with:
# volume=7.0, date=2024-06-28, plant != P_B1.

consumption_lookup_no_registered = consumption_merged_ready[
    (consumption_merged_ready["volume"] == 7.0) &
    (consumption_merged_ready["date"] == pd.to_datetime("2024-06-28")) &
    (consumption_merged_ready["plant"] != "P_B1")
]
consumption_lookup_no_registered

,date,plant,product,note,volume,cement_42_5r,cement_3a_42,cement_4av,arid_04,arid_8_14,arid_12_20,water,ad_complast_mr,ad_dynamon_sx,ad_aut_dynamon,ad_auramix,corrective_02
3309,2024-06-28,P_A1,HA-25-B-20-XC1,NaN,7.0,0.0,0.0,1938.0,7817.0,2425.0,3871.0,1057.0,17.4,10.65,0.0,0.0,0
3314,2024-06-28,P_A1,HA-30-F-20-X0,PILARES AMERIGO,7.0,0.0,0.0,2094.0,7616.0,2422.0,3764.0,1123.0,19.0,12.55,0.0,0.0,0
3315,2024-06-28,P_A1,HA-25-B-20,NaN,7.0,0.0,0.0,1938.0,7902.0,2489.0,3860.0,1058.0,17.4,10.70,0.0,0.0,0
3317,2024-06-28,P_A1,HA-25-B-20,NaN,7.0,0.0,0.0,1939.0,7748.0,2434.0,3921.0,1056.0,17.3,10.60,0.0,0.0,0
3322,2024-06-28,P_A1,HA-25-B-20-XC2,NaN,7.0,0.0,0.0,1935.0,7825.0,2441.0,3882.0,1058.0,17.4,10.60,0.0,0.0,0
3323,2024-06-28,P_A1,HA-30-F-20-X0,PILARES AMERIGO,7.0,0.0,0.0,2101.0,7531.0,2468.0,3695.0,1120.0,18.9,12.50,0.0,0.0,0
3324,2024-06-28,P_A1,HA-30-F-20-X0,PILARES AMERIGO,7.0,0.0,0.0,2101.0,7642.0,2445.0,3761.0,1120.0,18.9,12.55,0.0,0.0,0
3325,2024-06-28,P_A1,HA-30-F-20-X0,PILARES AMERIGO,7.0,0.0,0.0,2105.0,7576.0,2480.0,3731.0,1120.0,18.8,12.60,0.0,0.0,0
3331,2024-06-28,P_A1,HA-30-F-20-X0,PILARES AMERIGO,7.0,0.0,0.0,2099.0,7682.0,2461.0,3729.0,1121.0,18.8,12.55,0.0,0.0,0
3334,2024-06-28,P_A1,HA-25-B-20-XC2,NaN,7.0,0.0,0.0,1941.0,7830.0,2442.0,3893.0,1056.0,12.4,10.65,0.0,0.0,0


There are too many rows to find the product. 
We can treat it as a count of consumption, use the consumption average for its category, incorporate it into the error margin, or even delete it. I decide to count it as one more product.

In [16]:
# Cheking for "HA-30-B-IV-QB" products in products_years dataset. Arid is missing.
count = (products_years["product"] == "HA-30-B-IV-QB").sum()
total = len(products_years)
print(f"HA-30-B-IV-QB: {count} rows out of {total} ({count / total * 100:.2f}%)")
products_years[products_years["product"] == "HA-30-B-IV-QB"]

HA-30-B-IV-QB: 1 rows out of 9012 (0.01%)


,customer,worksite,plant,product,product_category,volume,billing_company,date
6695,COSMA_APOLLINARE_BASSANO,SENTINUM,P_B1,HA-30-B-IV-QB,regulatory_concrete,2.0,H,2025-07-03


In [17]:
# In this case we are able to conduct a better investigation.
consumption_lookup_ha30_biv_qb = consumption_merged_ready[
    (consumption_merged_ready["volume"] == 2.0) &
    (consumption_merged_ready["date"] == pd.to_datetime("2025-07-03")) &
    (consumption_merged_ready["plant"] == "P_B1")
]
consumption_lookup_ha30_biv_qb

,date,plant,product,note,volume,cement_42_5r,cement_3a_42,cement_4av,arid_04,arid_8_14,arid_12_20,water,ad_complast_mr,ad_dynamon_sx,ad_aut_dynamon,ad_auramix,corrective_02
18982,2025-07-03,P_B1,HA-30-B-20-IV-QB,NaN,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
18985,2025-07-03,P_B1,HA-30-B-20-IV,NaN,2.0,650.0,0.0,0.0,2100.0,490.0,1050.0,320.0,0.0,0.0,0.0,5.9,0


Even knowing that not all the loads are registered in the consumption dataset, `consumption_lookup_ha30_biv_qb` returned a match.

Given our consumption, I decided to assign arid-20 to the misspelling of the product name "HA-30-B-IV-QB" in the products_years dataset.

In [18]:
products_years["product"] = products_years["product"].replace("HA-30-B-IV-QB", "HA-30-B-20-IV-QB")

In [19]:
# Cheking for "HM-20-15" products in products_years dataset. Consistency is missing.
count = (products_years["product"] == "HM-20-15").sum()
total = len(products_years)
print(f"HM-20-15: {count} rows out of {total} ({count / total * 100:.2f}%)")
products_years[products_years["product"] == "HM-20-15"]

HM-20-15: 1 rows out of 9012 (0.01%)


,customer,worksite,plant,product,product_category,volume,billing_company,date
4380,PLINIO_GRACIANO,HIBERNIA_VELEIA,P_A,HM-20-15,regulatory_concrete,2.5,H,2024-12-23


In [20]:
consumption_lookup_hm20_15 = consumption_merged_ready[
    (consumption_merged_ready["volume"] == 2.5) &
    (consumption_merged_ready["date"] == pd.to_datetime("2024-12-23")) &
    (consumption_merged_ready["plant"] != "P_B1")
]
consumption_lookup_hm20_15

,date,plant,product,note,volume,cement_42_5r,cement_3a_42,cement_4av,arid_04,arid_8_14,arid_12_20,water,ad_complast_mr,ad_dynamon_sx,ad_aut_dynamon,ad_auramix,corrective_02


There is no load with 2.5 m3 in the consumption dataset. We cannot assume any consistency or assign any consumption either.

In [21]:
# checking for "MD" products in products_years dataset. 
count = (products_years["product"] == "MD").sum()
total = len(products_years)
print(f"MD: {count} rows out of {total} ({count / total * 100:.2f}%)")
products_years[products_years["product"] == "MD"]

MD: 1 rows out of 9012 (0.01%)


,customer,worksite,plant,product,product_category,volume,billing_company,date
1400,MILETO,ORVINIUM,P_B1,MD,cement_dosed_mixes,3.0,H,2024-04-19


In [22]:
consumption_lookup_md_by_volume = consumption_merged_ready[
    (consumption_merged_ready["volume"] == 3) &
    (consumption_merged_ready["date"] == pd.to_datetime("2024-04-19")) &
    (consumption_merged_ready["plant"] == "P_B1")
]
consumption_lookup_md_by_volume

,date,plant,product,note,volume,cement_42_5r,cement_3a_42,cement_4av,arid_04,arid_8_14,arid_12_20,water,ad_complast_mr,ad_dynamon_sx,ad_aut_dynamon,ad_auramix,corrective_02


In [23]:
# Search without volume.
consumption_lookup_md_by_product = consumption_merged_ready[
    (consumption_merged_ready["product"].str.startswith("MD")) &
    (consumption_merged_ready["date"] == pd.to_datetime("2024-04-19")) &
    (consumption_merged_ready["plant"] == "P_B1")
]
consumption_lookup_md_by_product

,date,plant,product,note,volume,cement_42_5r,cement_3a_42,cement_4av,arid_04,arid_8_14,arid_12_20,water,ad_complast_mr,ad_dynamon_sx,ad_aut_dynamon,ad_auramix,corrective_02


We can not observe any proof to estimate the real product.

---
---

### Comparison and exploration of unique products in both datasets.

In [24]:
prod_set = set(products_years["product"].dropna().unique())
cons_set = set(consumption_merged_ready["product"].dropna().unique())
all_unique_products = sorted(prod_set | cons_set)

all_unique_products_comparison = pd.DataFrame({
    "product":           all_unique_products,
    "in_products_years": ["✔" if p in prod_set else "" for p in all_unique_products],
    "in_consumption":    ["✔" if p in cons_set else "" for p in all_unique_products],
    "match":             ["✔" if (p in prod_set and p in cons_set) else "✘" for p in all_unique_products],
})

all_unique_products_comparison.to_excel(config["all_unique_products_comparison_path"], index=False)
all_unique_products_comparison

,product,in_products_years,in_consumption,match
0,C40-50-XC4,✔,,✘
1,C45-55-XC4,✔,,✘
2,CT-C12-F3,✔,✔,✔
3,CT-C16-F3,✔,✔,✔
4,D-100-B-15,,✔,✘
...,...,...,...,...
130,arid-04,✔,✔,✔
131,arid-15,,✔,✘
132,arid-20,✔,✔,✔
133,arid-25,✔,,✘


In [25]:
# Products that exist in the products dataset but have no matching record in consumption.
products_only_in_prod = sorted(set(products_years["product"].dropna().unique()) - set(consumption_merged_ready["product"].dropna().unique()))
display(products_only_in_prod)

['C40-50-XC4',
 'C45-55-XC4',
 'D-200',
 'D-250-B-15',
 'D-350',
 'D-400-B-14',
 'D-450-B-14',
 'H-100-B-15',
 'H-100-B-25',
 'H-175-B-25',
 'HA-25-B-20-XA1',
 'HA-25-F-20-XC3',
 'HA-30-F-15-QB',
 'HA-30-F-15-XC4-XA1',
 'HA-30-F-20-XC4',
 'HA-30-L-15',
 'HA-35-F-15-XC2',
 'HM-20-15',
 'HM-25-B-20-X0',
 'MD',
 'MD-150',
 'MD-200',
 'MD-250',
 'MD-400',
 'anhydrite',
 'arid',
 'arid-25',
 'no_registered']

In [26]:
# Group by total volume of each product per plant and month.
products_only_in_prod_df = (
    products_years[products_years["product"].isin(products_only_in_prod)]
    .assign(
        year=pd.to_datetime(products_years["date"]).dt.year,
        month=pd.to_datetime(products_years["date"]).dt.month,
    )
    .groupby(["product", "plant", "year", "month"])["volume"]
    .sum()
    .reset_index(name="total_volume_m3")
)
products_only_in_prod_df


,product,plant,year,month,total_volume_m3
0,C40-50-XC4,P_A,2024,4,6.0
1,C45-55-XC4,P_A,2024,4,8.0
2,D-200,P_A,2025,9,17.5
3,D-200,P_B1,2024,11,7.0
4,D-250-B-15,P_A,2024,8,4.0
...,...,...,...,...,...
125,arid,P_A,2025,6,2.0
126,arid,P_A,2025,10,7.0
127,arid,P_A,2025,11,7.0
128,arid-25,P_A,2025,2,38.0


In [27]:
products_only_in_prod_df.to_excel(config["products_only_in_prod_path"], index=False)

---
---

### Adding new plants to unmatch products.

In [28]:
# For products that only exist in products_years (not in consumption), reassign plant codes. Context owner reason (me).
# P_A  -> P_A3  |  P_B1 -> P_B2  |  P_C1 -> unchanged.

In [29]:
products_only_in_prod_df["product"].unique()

<StringArray>
[        'C40-50-XC4',         'C45-55-XC4',              'D-200',
         'D-250-B-15',              'D-350',         'D-400-B-14',
         'D-450-B-14',         'H-100-B-15',         'H-100-B-25',
         'H-175-B-25',     'HA-25-B-20-XA1',     'HA-25-F-20-XC3',
      'HA-30-F-15-QB', 'HA-30-F-15-XC4-XA1',     'HA-30-F-20-XC4',
         'HA-30-L-15',     'HA-35-F-15-XC2',           'HM-20-15',
      'HM-25-B-20-X0',                 'MD',             'MD-150',
             'MD-200',             'MD-250',             'MD-400',
          'anhydrite',               'arid',            'arid-25',
      'no_registered']
Length: 28, dtype: str

In [30]:
# P_A  -> P_A3  |  P_B1 -> P_B2  |  P_C1 -> unchanged.
unmatched_products = set(products_only_in_prod_df["product"].unique())

plant_remap = {"P_A": "P_A3", "P_B1": "P_B2"}

mask = products_years["product"].isin(unmatched_products)

products_years.loc[mask, "plant"] = (
    products_years.loc[mask, "plant"].replace(plant_remap)
)

# Verify result.
products_years.loc[mask, ["product", "plant"]].drop_duplicates().sort_values("plant")

,product,plant
4,MD-200,P_A3
7470,D-200,P_A3
7232,HA-30-L-15,P_A3
5818,H-100-B-25,P_A3
5482,HA-25-B-20-XA1,P_A3
5337,D-450-B-14,P_A3
5059,D-350,P_A3
4836,arid-25,P_A3
4380,HM-20-15,P_A3
3687,H-100-B-15,P_A3


In [31]:
# Checking 1out2.
# Frequency of each unmatched product in products_years.
unmatched_products = set(products_only_in_prod_df["product"].unique())

freq = (
    products_years[products_years["product"].isin(unmatched_products)]
    .groupby("product")
    .size()
    .reset_index(name="frequency")
    .sort_values("frequency", ascending=False)
)
freq

,product,frequency
21,MD-200,543
5,D-400-B-14,148
24,anhydrite,35
22,MD-250,18
25,arid,13
20,MD-150,8
2,D-200,4
12,HA-30-F-15-QB,3
9,H-175-B-25,3
26,arid-25,3


In [32]:
# Checking 2out2.
# Cross-check: frequency by product + plant vs total frequency.
freq_by_plant = (
    products_years[products_years["product"].isin(unmatched_products)]
    .groupby(["product", "plant"])
    .size()
    .reset_index(name="frequency")
    .sort_values(["product", "plant"])
)

freq_totals = freq_by_plant.groupby("product")["frequency"].sum().reset_index(name="sum_by_plant")
check = freq.merge(freq_totals, on="product")
check["match"] = check["frequency"] == check["sum_by_plant"]
check

,product,frequency,sum_by_plant,match
0,MD-200,543,543,True
1,D-400-B-14,148,148,True
2,anhydrite,35,35,True
3,MD-250,18,18,True
4,arid,13,13,True
5,MD-150,8,8,True
6,D-200,4,4,True
7,HA-30-F-15-QB,3,3,True
8,H-175-B-25,3,3,True
9,arid-25,3,3,True


---
---

### Product category. Improving category names and adding the product_category column to the consumption dataset.

In [33]:
products_years["product_category"].unique()

<StringArray>
[  'regulatory_concrete',     'regulatory_mortar',    'cement_dosed_mixes',
     'granular_material',        'modified_mixes', 'binder_based_material',
         'uncategorized']
Length: 7, dtype: str

In [34]:
# New categorization criteria. Based on EHE-8 due to old nomenclature. And based also on the new nomenclature Código estructural for the new formula verisons.
product_category_map = {
    # ── Structural Concrete (HA, HM, HF, C series)
    'HA': 'structural_concrete',
    'HM': 'structural_concrete',
    'HF': 'structural_concrete',
    'C40': 'structural_concrete',
    'C45': 'structural_concrete',

    # ── Non-Structural Concrete (H series, CT series)
    'H-': 'non_structural_concrete',
    'CT': 'non_structural_concrete',

    # ── Dosed Concrete (D series)
    'D-': 'dosed_concrete',

    # ── Dosed Mortar (MD series)
    'MD': 'dosed_mortar',

    # ── Granular Material
    'arid': 'granular_material',

    # ── Binder Material
    'anhydrite': 'binder_material',

    # ── Uncategorized
    'no_registered': 'uncategorized',
}

def assign_category(product: str) -> str:
    """Assigns a category to a product based on its prefix."""
    if pd.isna(product):
        return 'uncategorized'
    for prefix, category in product_category_map.items():
        if product.startswith(prefix):
            return category
    return 'uncategorized'

# Apply to dataframe
products_years['product_category'] = products_years['product'].apply(assign_category)

In [35]:
products_years["product_category"].unique()

<StringArray>
[    'structural_concrete', 'non_structural_concrete',
            'dosed_mortar',       'granular_material',
          'dosed_concrete',         'binder_material',
           'uncategorized']
Length: 7, dtype: str

In [36]:
# Is there any row with NaN in product_category in products_years dataset?
nan_category_count = products_years['product_category'].isna().sum()
nan_category_count

np.int64(0)

In [37]:
consumption_merged_ready['product_category'] = consumption_merged_ready['product'].apply(assign_category)

In [38]:
# Is there any row with NaN in product_category in consumption_merged_ready dataset?
nan_category_count = consumption_merged_ready['product_category'].isna().sum()
nan_category_count

np.int64(0)

---
---

###  Inserting ID columns.

In [39]:
# Each row is not a single delivery. It is the total volume dispatched for that day, aggregated by customer + worksite + product.
# In this case, setting delivery_id would be a misleading. We will use record_id instead, as a unique identifier for each row in products_years dataset.
products_years.insert(0, "record_id", range(len(products_years)))

In [40]:
# In consumption_merged_ready, each row belongs for a single load. Therefore, we can set load_id as a unique identifier for each row.
consumption_merged_ready.insert(0, "load_id", range(len(consumption_merged_ready)))

---
---

### Reordering columns.

In [41]:
consumption_merged_ready.head()

,load_id,date,plant,product,note,volume,cement_42_5r,cement_3a_42,cement_4av,arid_04,arid_8_14,arid_12_20,water,ad_complast_mr,ad_dynamon_sx,ad_aut_dynamon,ad_auramix,corrective_02,product_category
0,0,2024-12-27,P_A1,HL-150-B-20,NaN,4.0,602.0,0.0,0.0,4724.0,1590.0,1990.0,523.0,3.9,0.0,0.0,0.0,0,uncategorized
1,1,2024-12-27,P_A1,HL-150-B-20,NaN,7.0,1049.0,0.0,0.0,8440.0,2794.0,3493.0,911.0,6.9,0.0,0.0,0.0,0,uncategorized
2,2,2024-12-27,P_A1,HM-20-B-20,NaN,10.0,2022.0,0.0,0.0,11971.0,3492.0,5274.0,1288.0,16.2,0.0,0.0,0.0,0,structural_concrete
3,3,2024-12-27,P_A1,HA-30-B-20-XC2,NaN,6.0,1751.0,0.0,0.0,6651.0,1938.0,3391.0,958.0,11.4,8.9,0.0,0.0,0,structural_concrete
4,4,2024-12-27,P_A1,HL-150-B-20,NaN,5.0,749.0,0.0,0.0,5912.0,1987.0,2504.0,651.0,4.9,0.0,0.0,0.0,0,uncategorized


In [42]:
# Reorder product_category next to note.
consumption_merged_ready.columns.to_list

<bound method IndexOpsMixin.tolist of Index(['load_id', 'date', 'plant', 'product', 'note', 'volume', 'cement_42_5r',
       'cement_3a_42', 'cement_4av', 'arid_04', 'arid_8_14', 'arid_12_20',
       'water', 'ad_complast_mr', 'ad_dynamon_sx', 'ad_aut_dynamon',
       'ad_auramix', 'corrective_02', 'product_category'],
      dtype='str')>

In [43]:
# Date first. It is the primary dimension in time-series analysis and the most common filter in BI dashboards. Leading with date makes the dataset easier to scan, sort, and partition.
products_years = products_years[['record_id', 'date', 'customer', 'worksite', 'plant', 'product', 'product_category', 'volume', 'billing_company']]

In [44]:
products_years.head()

,record_id,date,customer,worksite,plant,product,product_category,volume,billing_company
0,0,2024-01-02,COSMA_VITRUVIO_PALLADIO,CASTRA_VEII_FIDENAE_CORFINIUM,P_A,HA-25-B-20-XC1,structural_concrete,10.0,H
1,1,2024-01-02,COSMA_Y_MODENA_HORTENSIA_ULISSE,PLACENTIA_BRUNDISIUM,P_A,HM-20-B-20,structural_concrete,7.0,H
2,2,2024-01-02,ETRURIA,CIRCUS_VELEIA,P_A,CT-C12-F3,non_structural_concrete,33.5,H
3,3,2024-01-02,FLORIANO_Y_ONORATO_HORTENSIA_CAPRI,RUSCINO_MEDULLIA,P_A,HA-25-B-20,structural_concrete,3.0,H
4,4,2024-01-02,NORCIA_XENOFONTE,DYRRACHIUM_DYRRACHIUM_SPOLETIUM_VEII_ZADAR,P_A3,MD-200,dosed_mortar,6.0,H


In [45]:
# Reorder product_category next to note.
consumption_merged_ready = consumption_merged_ready[['load_id', 'date', 'plant', 'product', 'product_category', 'note', 'volume', 'cement_42_5r', 'cement_3a_42', 'cement_4av', 'arid_04', 'arid_8_14', 'arid_12_20', 'water', 'ad_complast_mr', 'ad_dynamon_sx', 'ad_aut_dynamon', 'ad_auramix', 'corrective_02']]

---
---

### Sorting and cleanning the date column in consumption dataset.

By default, the dataset is sorted descending within each year (starting from the last working day and going backwards), and 2024 appears before 2025. The goal is to sort chronologically — from 2024-01-05 to 2025-12-30 — and strip the time component from the date column.

In [46]:
# Strip time component and sort chronologically ascending.
consumption_merged_ready["date"] = pd.to_datetime(consumption_merged_ready["date"]).dt.normalize()
consumption_merged_ready = consumption_merged_ready.sort_values("date").reset_index(drop=True)

# Reassign load_id to reflect the new chronological order.
consumption_merged_ready["load_id"] = range(len(consumption_merged_ready))

consumption_merged_ready["date"].head(), consumption_merged_ready["date"].tail()

(0   2024-01-02
 1   2024-01-02
 2   2024-01-02
 3   2024-01-02
 4   2024-01-02
 Name: date, dtype: datetime64[us],
 19838   2025-12-30
 19839   2025-12-30
 19840   2025-12-30
 19841   2025-12-30
 19842   2025-12-30
 Name: date, dtype: datetime64[us])

In [47]:
consumption_merged_ready.head()

,load_id,date,plant,product,product_category,note,volume,cement_42_5r,cement_3a_42,cement_4av,arid_04,arid_8_14,arid_12_20,water,ad_complast_mr,ad_dynamon_sx,ad_aut_dynamon,ad_auramix,corrective_02
0,0,2024-01-02,P_A1,HA-25-B-20,structural_concrete,NaN,3.0,0.0,0.0,831.0,3709.0,1158.0,1566.0,452.0,5.4,4.05,0.0,0.0,0
1,1,2024-01-02,P_A2,CT-C12-F3,non_structural_concrete,NaN,6.0,0.0,0.0,1510.0,10200.0,0.0,0.0,1289.0,0.0,0.00,16.5,0.0,0
2,2,2024-01-02,P_A2,CT-C12-F3,non_structural_concrete,NaN,6.0,0.0,0.0,1510.0,10270.0,0.0,0.0,1289.0,0.0,0.00,16.5,0.0,0
3,3,2024-01-02,P_A2,MD-200-P-4,dosed_mortar,M-5 CON,6.0,0.0,0.0,1202.0,11450.0,0.0,0.0,833.0,7.2,0.00,0.0,0.0,0
4,4,2024-01-02,P_A2,CT-C12-F3,non_structural_concrete,NaN,6.0,0.0,0.0,1508.0,10190.0,0.0,0.0,1289.0,0.0,0.00,16.5,0.0,0


In [48]:
consumption_merged_ready.dtypes

load_id                      int64
date                datetime64[us]
plant                          str
product                        str
product_category               str
note                           str
volume                     float64
cement_42_5r               float64
cement_3a_42               float64
cement_4av                 float64
arid_04                    float64
arid_8_14                  float64
arid_12_20                 float64
water                      float64
ad_complast_mr             float64
ad_dynamon_sx              float64
ad_aut_dynamon             float64
ad_auramix                 float64
corrective_02                int64
dtype: object

In [49]:
products_years.dtypes

record_id             int64
date                    str
customer                str
worksite                str
plant                   str
product                 str
product_category        str
volume              float64
billing_company         str
dtype: object

In [ ]:
sywhds

In [ ]:
#### Adding zone_column. P_A, P_B, P_C.


---
---

### Exporting

#### Exporting final datasets, fixing date format.

In [51]:
# Exporting both datasets with the new modification in this notebook.
# .dt.date converts datetime64 to Python date objects — openpyxl writes pure DATE cells (no hh:mm:ss in Excel, SQL or Power BI).
export_consumption = consumption_merged_ready.copy()
export_consumption["date"] = export_consumption["date"].dt.date
export_consumption.to_csv(config["consumption_db_csv_path"], index=False)

In [55]:
export_consumption.dtypes

load_id               int64
date                 object
plant                   str
product                 str
product_category        str
note                    str
volume              float64
cement_42_5r        float64
cement_3a_42        float64
cement_4av          float64
arid_04             float64
arid_8_14           float64
arid_12_20          float64
water               float64
ad_complast_mr      float64
ad_dynamon_sx       float64
ad_aut_dynamon      float64
ad_auramix          float64
corrective_02         int64
dtype: object

*Note from sanity check, from the beginning.*

date is str in products_years, but datetime in consumption_merged_ready. We need to convert date in products_years to datetime.
However, I decided finally convert it just for the exportation for the following reasons:

1. In some ahead steps, the dataframe became again to the original format.
2. There is no need to convert the format now.

*Note in "learn_by_issues" file about how pandas reads date data types and how to convert it.*

In [57]:
export_products = products_years.copy()
export_products["date"] = pd.to_datetime(export_products["date"]).dt.date
export_products.to_csv(config["daily_deliveries_db_csv_path"], index=False)

In [58]:
export_consumption.dtypes

load_id               int64
date                 object
plant                   str
product                 str
product_category        str
note                    str
volume              float64
cement_42_5r        float64
cement_3a_42        float64
cement_4av          float64
arid_04             float64
arid_8_14           float64
arid_12_20          float64
water               float64
ad_complast_mr      float64
ad_dynamon_sx       float64
ad_aut_dynamon      float64
ad_auramix          float64
corrective_02         int64
dtype: object

---

#### Exporting final datasets to CSV (MySQL-ready format).

Dates are formatted as `YYYY-MM-DD` strings (ISO 8601), which MySQL interprets natively as `DATE` columns.
UTF-8 encoding, no index, and empty strings for null values — compatible with `LOAD DATA INFILE` and standard `INSERT` workflows.

In [53]:
# Export consumption dataset to CSV.
# .dt.strftime ensures the date column is written as a plain 'YYYY-MM-DD' string.
# MySQL DATE columns accept this format directly with no additional casting needed.
export_consumption_csv = consumption_merged_ready.copy()
export_consumption_csv["date"] = export_consumption_csv["date"].dt.strftime("%Y-%m-%d")
export_consumption_csv.to_csv(config["consumption_db_csv_path"], index=False, encoding="utf-8")

In [59]:
export_consumption_csv.dtypes

load_id               int64
date                    str
plant                   str
product                 str
product_category        str
note                    str
volume              float64
cement_42_5r        float64
cement_3a_42        float64
cement_4av          float64
arid_04             float64
arid_8_14           float64
arid_12_20          float64
water               float64
ad_complast_mr      float64
ad_dynamon_sx       float64
ad_aut_dynamon      float64
ad_auramix          float64
corrective_02         int64
dtype: object

In [54]:
# Export products dataset to CSV.
# products_years["date"] is still a string object (str dtype), so pd.to_datetime is applied
# before strftime to guarantee a consistent 'YYYY-MM-DD' output regardless of source format.
export_products_csv = products_years.copy()
export_products_csv["date"] = pd.to_datetime(export_products_csv["date"]).dt.strftime("%Y-%m-%d")
export_products_csv.to_csv(config["daily_deliveries_db_csv_path"], index=False, encoding="utf-8")

In [60]:
export_products.dtypes

record_id             int64
date                 object
customer                str
worksite                str
plant                   str
product                 str
product_category        str
volume              float64
billing_company         str
dtype: object